<a href="https://colab.research.google.com/github/1816x/Algoritmos-Aprendizaje-Automatico/blob/main/Practica_Tema_12_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practica Tema 12

**Clase:** Fundamentos Algoritmos de Aprendizaje Automatico  
**Tema:** Sesgos, Equidad y Explicabilidad

## reto 1. entrenamiento del modelo base y particion de datos de auditoria

Se entrena un modelo random forest y se generan predicciones sobre el conjunto de prueba para preparar la auditoria.

In [1]:
from sklearn.ensemble import RandomForestClassifier as random_forest_classifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np

x, y = make_classification(
    n_samples=1000,
    n_features=10,
    weights=[0.7, 0.3],
    random_state=42
)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

model = random_forest_classifier(
    n_estimators=100,
    random_state=42
)

model.fit(x_train, y_train)
y_pred = model.predict(x_test)

print("base model trained successfully")
print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)


base model trained successfully
x_train shape: (800, 10)
x_test shape: (200, 10)
y_test shape: (200,)


netrenas el modelo que después se auditara. genera datos de clasificación, separas entrenamiento y prueba, entrenas un Random Forest y haces predicciones. se necesita primero  un modelo funcionando antes de poder revisar si es justo o explicable

## reto 2. auditoria manual de metricas de equidad por subgrupo

Se simula un atributo sensible binario y se calculan TPR y FPR por separado para cada grupo.

In [2]:
rng = np.random.default_rng(42)
sensitive_attribute = rng.choice(["group_a", "group_b"], size=len(x_test))

audit_df = pd.DataFrame({
    "y_true": y_test,
    "y_pred": y_pred,
    "group": sensitive_attribute
})

def calculate_rates(group_data):
    tn, fp, fn, tp = confusion_matrix(
        group_data["y_true"],
        group_data["y_pred"],
        labels=[0, 1]
    ).ravel()

    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

    return tpr, fpr

fairness_results = []

for group_name in ["group_a", "group_b"]:
    group_data = audit_df[audit_df["group"] == group_name]
    tpr, fpr = calculate_rates(group_data)

    fairness_results.append({
        "group": group_name,
        "tpr": tpr,
        "fpr": fpr
    })

fairness_df = pd.DataFrame(fairness_results)
fairness_df


,group,tpr,fpr
0,group_a,0.833333,0.014493
1,group_b,0.850000,0.065574


revisas si el modelo se comporta igual para dos grupos distintos. Comparas TPR y FPR. El TPR dice qué tan bien detecta correctamente los casos positivos. El FPR dice cuántas veces marca como positivo algo que realmente era negativo. Si un grupo tiene resultados mucho peores que otro, puede haber un problema de equidad.

Si TPR y FPR cambian mucho entre grupos, el modelo se comporta de forma diferente para cada subgrupo y esa diferencia debe investigarse.

## reto 3. analisis de variables proxy y fuga de sesgo historico

Una variable proxy puede estar relacionada con un atributo sensible aunque dicho atributo no se use directamente. Esto puede permitir que el modelo reproduzca sesgos de forma indirecta.

In [3]:
numeric_sensitive = (sensitive_attribute == "group_b").astype(int)

proxy_df = pd.DataFrame(x_test)
proxy_df.columns = [f"feature_{i}" for i in range(proxy_df.shape[1])]
proxy_df["sensitive_attribute"] = numeric_sensitive

correlations = proxy_df.corr()["sensitive_attribute"].drop("sensitive_attribute")
correlations = correlations.abs().sort_values(ascending=False)

print("feature correlations with sensitive attribute:")
print(correlations)


feature correlations with sensitive attribute:
feature_9    0.104718
feature_6    0.071407
feature_8    0.067334
feature_2    0.046073
feature_5    0.027090
feature_3    0.025547
feature_7    0.022357
feature_4    0.013352
feature_1    0.012894
feature_0    0.008655
Name: sensitive_attribute, dtype: float64


aquí buscas sesgos indirectos. Aunque elimines del modelo una variable sensible, como género o grupo social, otra variable podría estar muy relacionada con ella y funcionar como sustituto o proxy. Por eso revisas correlaciones: para detectar variables que podrían estar filtrando indirectamente información sensible.

## reto 4. explicabilidad local y global con shap/lime

La explicacion global muestra que variables influyen mas en el modelo en general. La explicacion local busca entender una prediccion individual.

In [4]:
feature_importance_df = pd.DataFrame({
    "feature": [f"feature_{i}" for i in range(x_train.shape[1])],
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("global explanation using feature importance:")
print(feature_importance_df)

sample_index = 0
sample = x_test[sample_index].reshape(1, -1)
sample_prediction = model.predict(sample)[0]
sample_probability = model.predict_proba(sample)[0]

print("\nlocal observation index:", sample_index)
print("prediction:", sample_prediction)
print("class probabilities:", sample_probability)


global explanation using feature importance:
     feature  importance
6  feature_6    0.382438
2  feature_2    0.214366
8  feature_8    0.120637
0  feature_0    0.108631
5  feature_5    0.034819
7  feature_7    0.030252
1  feature_1    0.028861
3  feature_3    0.028341
4  feature_4    0.027337
9  feature_9    0.024317

local observation index: 0
prediction: 0
class probabilities: [0.92 0.08]


aquí intentas entender por qué el modelo toma sus decisiones. Una explicación global te dice qué variables influyen más en el modelo en general. Una explicación local te dice por qué una observación específica recibió cierta predicción. SHAP y LIME sirven precisamente para hacer más entendibles modelos complejos como Random Forest.

## conclusion

La auditoria de un modelo no debe limitarse a medir precision. Tambien hay que comparar su comportamiento entre grupos, detectar variables proxy y explicar como se generan las decisiones.